# Overzoom — super-resolution past the deepest imagery level, a proof of concept

Companion of `imagery-tile-enhancement.md` (§1.4, §3.2–3.3, §7 experiment 4). Nothing here touches the engine: it reads
whole PMTiles archives of an imagery layer, turns them into a tile dataset, and measures what a super-resolution model
does to tiles that are already at their deepest level.

**The trick that makes it measurable.** The pyramid holds real `(z, z+1)` pairs. Predict a z19 tile from its z18 parent
and compare with the true z19 tile: same sensor, same mosaic, same JPEG, nothing invented to evaluate. Two inputs are
measured:

- **real**: the z18 tile as served (the honest case — what the client actually has above z19);
- **synthetic**: the z19 tile reduced 2×2 (the textbook case; the gap between the two says how far the served parent is
  from a clean reduction of its children).

Then the models run past z19 — z20 and z21, where no truth exists — and the mosaics are **opened and looked at**.

Method, each time: what is supposed (plan), what is run (do), what the numbers say (check), what follows (act).

**Data.** Archives are read through read-only presigned URLs listed in a JSON file (`[{key, url, expires_at}]`, the
zone manifest first) that never enters the repository. Imagery tiles and outputs never enter it either.

## 0. Environment

Runs on Colab (GPU runtime) or on a GPU VM with Jupyter. `torch` is assumed present (it is on Colab and on the deep
learning VM images); the rest is installed here. Model weights come from the Hugging Face Hub at run time, pinned to
a revision, and are never stored next to the code.

In [ ]:
import os, sys, subprocess, json, math, time, random, io
from pathlib import Path

def pip(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)

pip('pmtiles', 'spandrel', 'lpips', 'scikit-image', 'requests', 'pillow', 'pandas', 'matplotlib')

import numpy as np, pandas as pd, requests, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from skimage.metrics import structural_similarity

IN_COLAB = 'google.colab' in sys.modules
DEV = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('device:', DEV, torch.cuda.get_device_name(0) if DEV == 'cuda' else '')

ROOT = Path(os.environ.get('POC_ROOT', '/content/overzoom' if IN_COLAB else 'overzoom'))
ARCHIVES, TILES, WEIGHTS, OUT = (ROOT / d for d in ('archives', 'tiles', 'weights', 'out'))
for d in (ARCHIVES, TILES, WEIGHTS, OUT):
    d.mkdir(parents=True, exist_ok=True)

URLS_JSON = Path(os.environ.get('POC_URLS', ROOT / 'presigned.json'))
if not URLS_JSON.exists() and IN_COLAB:
    from google.colab import files
    print('Waiting for the presigned-URL JSON file: click "Choose files" below.\n'
          'Without it, interrupt this cell (nothing else in the notebook can run).')
    up = files.upload()
    if not up:
        raise SystemExit('no file uploaded')
    URLS_JSON.write_bytes(next(iter(up.values())))
if not URLS_JSON.exists():
    raise SystemExit(f'missing {URLS_JSON}: the list of presigned URLs ([{{key, url, expires_at}}])')
entries = json.loads(URLS_JSON.read_text())
print(len(entries), 'objects, expiring', sorted({e.get('expires_at', '?') for e in entries}))

## 1. Download whole archives, merge them into one tile set

The zone manifest lists its live archives newest first; a tile present in several archives is taken from the newest.
Archives land under `archives/` mirroring their store key, so a re-run skips what is already there.

In [ ]:
def download(url, dst):
    if dst.exists() and dst.stat().st_size > 0:
        return dst
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_suffix(dst.suffix + '.part')
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(tmp, 'wb') as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
    tmp.rename(dst)
    return dst

t0 = time.time()
local = {e['key']: download(e['url'], ARCHIVES / e['key']) for e in entries}
total = sum(p.stat().st_size for p in local.values())
print(f'{len(local)} objects, {total / 1e6:.0f} MB in {time.time() - t0:.0f} s')

manifests = [k for k in local if k.endswith('manifest.json')]
order = []
for m in manifests:
    order += [a['key'] for a in json.loads(local[m].read_text()).get('archives', [])]
order += [k for k in local if k.endswith('.pmtiles') and k not in order]
order = [k for k in order if k in local]
print('merge order (newest first):', *[Path(k).name for k in order], sep='\n  ')

In [ ]:
from pmtiles.reader import MmapSource, Reader, all_tiles

tiles, rows = {}, []
for key in order:
    with open(local[key], 'rb') as f:
        src = MmapSource(f)
        h = Reader(src).header()
        n = 0
        for (z, x, y), data in all_tiles(src):
            n += 1
            if (z, x, y) not in tiles:
                tiles[(z, x, y)] = data
                rows.append((z, x, y, Path(key).name, len(data)))
        print(f"{Path(key).name}: {n} tiles, z{h['min_zoom']}-{h['max_zoom']}, type {h['tile_type']}")

index = pd.DataFrame(rows, columns=['z', 'x', 'y', 'archive', 'bytes'])
for (z, x, y), data in tiles.items():
    p = TILES / str(z) / str(x) / f'{y}.jpg'
    if not p.exists():
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_bytes(data)
index.to_csv(TILES / 'index.csv', index=False)
print(index.groupby('z').agg(tiles=('x', 'size'), mean_kb=('bytes', lambda b: round(b.mean() / 1024, 1))))

In [ ]:
zmax = int(index.z.max())
cov = index[index.z == zmax]
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(cov.x, cov.y, s=2, marker='s', color='#3b6ea5')
ax.invert_yaxis(); ax.set_aspect('equal')
ax.set_title(f'z{zmax} coverage, {len(cov)} tiles', loc='left')
plt.show()

## 2. Pixel windows across tiles

Every input is a window in global pixel coordinates at one zoom (`tile * 256 + local`), so a halo around a tile is read
from its real neighbours rather than invented. A window touching a missing tile returns `None` and the sample is
skipped — no padding pretends to be ground.

In [ ]:
TS = 256
_cache = {}

def tile_rgb(z, x, y):
    k = (z, x, y)
    if k not in _cache:
        data = tiles.get(k)
        _cache[k] = None if data is None else np.asarray(Image.open(io.BytesIO(data)).convert('RGB'), np.float32) / 255
    return _cache[k]

def window(z, gx, gy, w, h):
    out = np.empty((h, w, 3), np.float32)
    for ty in range(gy // TS, (gy + h - 1) // TS + 1):
        for tx in range(gx // TS, (gx + w - 1) // TS + 1):
            t = tile_rgb(z, tx, ty)
            if t is None:
                return None
            x0, y0 = max(gx, tx * TS), max(gy, ty * TS)
            x1, y1 = min(gx + w, (tx + 1) * TS), min(gy + h, (ty + 1) * TS)
            out[y0 - gy:y1 - gy, x0 - gx:x1 - gx] = t[y0 - ty * TS:y1 - ty * TS, x0 - tx * TS:x1 - tx * TS]
    return out

def to_t(a):
    return torch.from_numpy(a).permute(2, 0, 1)[None].to(DEV)

def to_np(t):
    return t[0].clamp(0, 1).permute(1, 2, 0).float().cpu().numpy()

def down2(t):
    # The pyramid's reduction is unknown for this source; a 2x2 box is the assumption, and section 5 measures it.
    return F.avg_pool2d(t, 2)

def up_bicubic(t, s=2):
    return F.interpolate(t, scale_factor=s, mode='bicubic', align_corners=False)

## 3. The methods

- `bicubic`, `lanczos`: the floor — what a GPU sampler or a resampler gives for free.
- `sharpen`: bicubic then an unsharp mask — the "supernetteté" without learning.
- `realesrgan-x2`, `bsrgan-x2`, `swinir-real-x2`: trained on synthetic real-world degradations (blur, noise, JPEG),
  the right family for web tiles; GAN-flavoured, so sharper and more inventive.
- `swinir-classic-x2`, `dat-x2`: trained on clean bicubic reduction — fidelity-oriented, the "little invention" end.
- for ×4 only (section 8): `realesrgan-x4`, and `swin2sr-jpeg-x4`, trained on compressed inputs.

Each learned method also runs with **pyramid consistency** (`+cem`): iterative back-projection so that the output,
reduced 2×2, gives back the input exactly — the CEM principle (Bahat & Michaeli 2020) with a known kernel. It can only
remove what contradicts the parent; it never adds detail.

In [ ]:
pip('huggingface_hub')
from huggingface_hub import hf_hub_download
from spandrel import ModelLoader

# Weights from the Hugging Face Hub, pinned to a revision so a re-run loads the same bytes. The architectures' own
# licences: Real-ESRGAN BSD-3-Clause, BSRGAN / SwinIR / Swin2SR / DAT Apache-2.0 (the repackaging repo says openrail;
# read the original before anything leaves the PoC).
UPSCALERS = ('licyk/sd-upscaler-models', '3a2094a57f456c41b79e5a4d196f2222c9940b89')
MODELS = {
    'realesrgan-x2': ('ai-forever/Real-ESRGAN', '8110204ebf8d25c031b66c26c2d1098aa831157e', 'RealESRGAN_x2.pth'),
    'bsrgan-x2': (*UPSCALERS, 'ESRGAN/BSRGANx2.pth'),
    'swinir-real-x2': (*UPSCALERS, 'SwinIR/003_realSR_BSRGAN_DFO_s64w8_SwinIR-M_x2_GAN-with-dict-keys-params-and-params_ema.pth'),
    'swinir-classic-x2': (*UPSCALERS, 'SwinIR/001_classicalSR_DF2K_s64w8_SwinIR-M_x2.pth'),
    'dat-x2': (*UPSCALERS, 'DAT/DAT_x2.pth'),
    'realesrgan-x4': (*UPSCALERS, 'RealESRGAN/RealESRGAN_x4plus.pth'),
    'swin2sr-jpeg-x4': (*UPSCALERS, 'SwinIR/Swin2SR_CompressedSR_X4_48.pth'),
}
nets = {}
for name, (repo, rev, fname) in MODELS.items():
    try:
        p = hf_hub_download(repo, fname, revision=rev, cache_dir=str(WEIGHTS))
        d = ModelLoader().load_from_file(p).to(DEV).eval()
        if DEV == 'cuda' and d.supports_half:
            d = d.half()
        nets[name] = d
        print(f'{name}: {d.architecture.name} x{d.scale}')
    except Exception as e:
        print(f'{name}: skipped ({e})')

@torch.no_grad()
def run_net(d, t):
    m = getattr(d.size_requirements, 'multiple_of', 1) or 1
    h, w = t.shape[-2:]
    ph, pw = (-h) % m, (-w) % m
    x = F.pad(t, (0, pw, 0, ph), mode='reflect') if ph or pw else t
    x = x.half() if next(d.model.parameters()).dtype == torch.float16 else x
    y = d(x).float()
    return y[..., :h * d.scale, :w * d.scale]

def sharpen(t, amount=0.8, sigma=1.0):
    k = int(2 * round(3 * sigma) + 1)
    g = torch.exp(-((torch.arange(k, device=t.device) - k // 2) ** 2) / (2 * sigma ** 2)); g = g / g.sum()
    blur = F.conv2d(F.pad(t, (k // 2,) * 4, mode='reflect'), g.view(1, 1, 1, k).repeat(3, 1, 1, 1), groups=3)
    blur = F.conv2d(blur, g.view(1, 1, k, 1).repeat(3, 1, 1, 1), groups=3)
    return t + amount * (t - blur)

def lanczos(t):
    a = to_np(t)
    im = Image.fromarray((a * 255 + 0.5).astype(np.uint8)).resize((a.shape[1] * 2, a.shape[0] * 2), Image.Resampling.LANCZOS)
    return to_t(np.asarray(im, np.float32) / 255)

def cem(lr, sr, iters=20):
    # Back-projection: sr <- sr + up(lr - down(sr)); converges to down(sr) == lr for the box kernel.
    for _ in range(iters):
        sr = sr + up_bicubic(lr - down2(sr))
    return sr

METHODS = {
    'bicubic': lambda t: up_bicubic(t),
    'lanczos': lanczos,
    'sharpen': lambda t: sharpen(up_bicubic(t)),
}
for name, d in nets.items():
    if d.scale == 2:
        METHODS[name] = (lambda d: lambda t: run_net(d, t))(d)
        METHODS[name + '+cem'] = (lambda d: lambda t: cem(t, run_net(d, t)))(d)
print(list(METHODS))

## 4. Bench: predict z19 from z18, compare with the true z19

Plan: a learned ×2 beats bicubic on LPIPS by a margin an eye would see, and `+cem` keeps it while restoring the parent.
Do: a fixed random sample of z19 tiles whose whole context exists; each prediction is made on a window with a halo of
`H` parent pixels read from the neighbours, then cropped to the tile.

Metrics against the true tile: PSNR and SSIM (fidelity), LPIPS (perceived distance, the one that tracks usefulness
downstream per the literature), and **parent PSNR**: the prediction reduced 2×2 against its input — how much the method
contradicts the tile the client already shows.

In [ ]:
pip('lpips')
import lpips
lp = lpips.LPIPS(net='alex', verbose=False).to(DEV)

H = 32            # halo, in input pixels
N = int(os.environ.get('POC_SAMPLES', 200))
z = zmax
cands = [(x, y) for (zz, x, y) in tiles if zz == z]
random.Random(17501).shuffle(cands)

def psnr(a, b):
    return float(10 * np.log10(1.0 / max(np.mean((a - b) ** 2), 1e-12)))

samples, bench = [], []
for (x, y) in cands:
    if len(samples) >= N:
        break
    gt = tile_rgb(z, x, y)
    real = window(z - 1, x * 128 - H, y * 128 - H, 128 + 2 * H, 128 + 2 * H)
    big = window(z, x * TS - 2 * H, y * TS - 2 * H, TS + 4 * H, TS + 4 * H)
    if gt is None or real is None or big is None:
        continue
    synth = to_np(down2(to_t(big)))
    samples.append((x, y, real, synth))

print(len(samples), 'samples with full context')
for kind_i, kind in enumerate(('real', 'synthetic')):
    for name, f in METHODS.items():
        t0 = time.time()
        for (x, y, real, synth) in samples:
            lr = to_t(real if kind == 'real' else synth)
            with torch.no_grad():
                sr = f(lr).clamp(0, 1)
                out = sr[..., 2 * H:2 * H + TS, 2 * H:2 * H + TS]
                gt_t = to_t(tile_rgb(z, x, y))
                l = float(lp(out * 2 - 1, gt_t * 2 - 1))
                par = psnr(to_np(down2(out)), to_np(lr[..., H:H + 128, H:H + 128]))
            o, g = to_np(out), to_np(gt_t)
            bench.append(dict(input=kind, method=name, x=x, y=y, psnr=psnr(o, g),
                              ssim=structural_similarity(o, g, channel_axis=2, data_range=1.0),
                              lpips=l, parent_psnr=par))
        if DEV == 'cuda':
            torch.cuda.synchronize()
        print(f'{kind:9s} {name:22s} {1000 * (time.time() - t0) / len(samples):7.1f} ms/tile')

bench = pd.DataFrame(bench)
bench.to_csv(OUT / 'bench.csv', index=False)
summary = bench.groupby(['input', 'method'])[['psnr', 'ssim', 'lpips', 'parent_psnr']].mean().round(4)
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, kind in zip(axes, ('real', 'synthetic')):
    s = summary.loc[kind].sort_values('lpips', ascending=False)
    ax.barh(s.index, s.lpips, color='#3b6ea5', height=0.6)
    for i, v in enumerate(s.lpips):
        ax.text(v, i, f' {v:.3f}', va='center', fontsize=9)
    ax.set_title(f'LPIPS to the true z{z} tile — {kind} z{z - 1} input (lower is better)', loc='left', fontsize=10)
    for sp in ('top', 'right'):
        ax.spines[sp].set_visible(False)
plt.tight_layout(); plt.savefig(OUT / 'bench-lpips.png', dpi=130); plt.show()

## 5. Is the served parent a 2×2 box reduction of its children?

`+cem` assumes it. If the served z18 is far from `down2(z19 children)` — another flight, another kernel, another JPEG
pass — then forcing consistency with it is forcing consistency with something else, and the real/synthetic gap in §4
is mostly this.

In [ ]:
gaps = []
for (x, y, real, synth) in samples[:100]:
    gaps.append(psnr(real[H:H + 128, H:H + 128], synth[H:H + 128, H:H + 128]))
print(f'served z{z - 1} vs box-reduced z{z}: PSNR median {np.median(gaps):.1f} dB, '
      f'p10 {np.percentile(gaps, 10):.1f}, p90 {np.percentile(gaps, 90):.1f}')

## 6. Look at it — the same tiles, side by side

A score lies; the picture does not. Six samples spread over the LPIPS range of the bicubic floor, each row:
true z19, bicubic, then each learned method with and without consistency.

In [ ]:
show = ['bicubic', 'sharpen'] + [m for m in METHODS if m not in ('bicubic', 'lanczos', 'sharpen')]
b = bench[(bench.input == 'real') & (bench.method == 'bicubic')].sort_values('lpips')
pick = b.iloc[np.linspace(0, len(b) - 1, 6).astype(int)][['x', 'y']].values.tolist()
crop = 128  # zoom on the tile centre so detail is visible on screen
fig, axes = plt.subplots(len(pick), len(show) + 1, figsize=(2.4 * (len(show) + 1), 2.4 * len(pick)))
for r, (x, y) in enumerate(pick):
    real = next(s[2] for s in samples if (s[0], s[1]) == (x, y))
    lr = to_t(real)
    c0 = (TS - crop) // 2
    axes[r, 0].imshow(tile_rgb(z, x, y)[c0:c0 + crop, c0:c0 + crop]); axes[r, 0].set_title('true' if r == 0 else '')
    for c, m in enumerate(show, 1):
        with torch.no_grad():
            o = to_np(METHODS[m](lr).clamp(0, 1)[..., 2 * H:2 * H + TS, 2 * H:2 * H + TS])
        axes[r, c].imshow(o[c0:c0 + crop, c0:c0 + crop])
        if r == 0:
            axes[r, c].set_title(m, fontsize=8)
for a in axes.flat:
    a.axis('off')
plt.tight_layout(); plt.savefig(OUT / 'bench-side-by-side.png', dpi=110); plt.show()

## 7. Seams: tile by tile versus with a halo

A server renders `(z, x, y)` alone. Without a halo, a convolutional model sees a hard border and the output jumps at
every tile edge. Seam score: mean absolute step across the tile boundaries of a mosaic, divided by the mean step between
neighbouring interior columns. 1 is seamless; above 1 the grid shows. The source mosaic's own score is the floor: its
tiles are separate JPEGs, so their edges already step a little (about 1.4 on a synthetic check).

In [ ]:
def superzoom(z0, x0, y0, n, f, scale, halo):
    """Upscale an n x n block of z0 tiles tile by tile, each from a window with `halo` pixels of real neighbours."""
    S = TS * scale
    out = np.zeros((n * S, n * S, 3), np.float32)
    for j in range(n):
        for i in range(n):
            w = window(z0, (x0 + i) * TS - halo, (y0 + j) * TS - halo, TS + 2 * halo, TS + 2 * halo)
            with torch.no_grad():
                o = f(to_t(w)).clamp(0, 1)
            out[j * S:(j + 1) * S, i * S:(i + 1) * S] = to_np(o[..., halo * scale:halo * scale + S,
                                                               halo * scale:halo * scale + S])
    return out

def seam_score(m, S):
    d = np.abs(np.diff(m, axis=1)).mean(axis=(0, 2))
    edges = np.arange(S - 1, m.shape[1] - 1, S)
    inner = np.setdiff1d(np.arange(len(d)), np.concatenate([edges - 1, edges, edges + 1]))
    return float(d[edges].mean() / d[inner].mean())

def block_ok(z0, x0, y0, n, halo=64):
    return window(z0, x0 * TS - halo, y0 * TS - halo, n * TS + 2 * halo, n * TS + 2 * halo) is not None

def texture(x, y):
    t = tile_rgb(z, x, y)
    return float(np.abs(np.diff(t.mean(2), axis=0)).mean()) if t is not None else 0

blocks = sorted(((texture(x, y), x, y) for (zz, x, y) in tiles if zz == z), reverse=True)
x0, y0 = next((x, y) for _, x, y in blocks[:400] if block_ok(z, x, y, 3))
print(f'block: z{z} {x0},{y0} (3x3 tiles)')

seams = []
for m in [k for k in METHODS if k != 'lanczos']:
    for halo in (0, 32):
        mos = superzoom(z, x0, y0, 3, METHODS[m], 2, halo)
        seams.append(dict(method=m, halo=halo, seam=seam_score(mos, 2 * TS)))
# The source's own tile edges are not seamless (each tile is a separate JPEG): its score is the floor to compare with.
src_score = seam_score(window(z, x0 * TS, y0 * TS, 3 * TS, 3 * TS), TS)
seams += [dict(method=f'source z{z}', halo=h, seam=src_score) for h in (0, 32)]
seams = pd.DataFrame(seams).pivot(index='method', columns='halo', values='seam').round(3)
seams.to_csv(OUT / 'seams.csv'); seams

## 8. Past the last level: z20 and z21

No truth here — this is the product question. The block above, served at z20 (×2) and z21 (×4: the ×4 model, and the
×2 model applied twice). Saved as PNG under `out/`, then **opened**.

In [ ]:
def save(a, name):
    p = OUT / name
    Image.fromarray((a * 255 + 0.5).astype(np.uint8)).save(p)
    return p

saved = [save(superzoom(z, x0, y0, 3, METHODS['bicubic'], 2, 32), f'z{z + 1}-bicubic.png')]
for m in [k for k in METHODS if k.endswith('-x2') or k.endswith('-x2+cem')]:
    saved.append(save(superzoom(z, x0, y0, 3, METHODS[m], 2, 32), f'z{z + 1}-{m}.png'))

saved.append(save(superzoom(z, x0, y0, 2, lambda t: up_bicubic(t, 4), 4, 32), f'z{z + 2}-bicubic.png'))
for m in ('realesrgan-x4', 'swin2sr-jpeg-x4'):
    if m in nets:
        saved.append(save(superzoom(z, x0, y0, 2, (lambda d: lambda t: run_net(d, t))(nets[m]), 4, 32),
                          f'z{z + 2}-{m}.png'))
for m in ('realesrgan-x2', 'swinir-real-x2'):
    if m in METHODS:
        twice = (lambda f: lambda t: f(f(t)))(METHODS[m + '+cem'])
        saved.append(save(superzoom(z, x0, y0, 2, twice, 4, 32), f'z{z + 2}-{m}+cem-twice.png'))
print(*saved, sep='\n')

In [ ]:
crop = 384
names = [p for p in saved if p.name.startswith(f'z{z + 1}')]
fig, axes = plt.subplots(1, len(names), figsize=(4 * len(names), 4.4))
for a, p in zip(np.atleast_1d(axes), names):
    im = np.asarray(Image.open(p))
    c = (im.shape[0] - crop) // 2
    a.imshow(im[c:c + crop, c:c + crop]); a.set_title(p.stem, fontsize=8); a.axis('off')
plt.tight_layout(); plt.savefig(OUT / f'z{z + 1}-side-by-side.png', dpi=110); plt.show()

names = [p for p in saved if p.name.startswith(f'z{z + 2}')]
fig, axes = plt.subplots(1, len(names), figsize=(4 * len(names), 4.4))
for a, p in zip(np.atleast_1d(axes), names):
    im = np.asarray(Image.open(p))
    c = (im.shape[0] - crop) // 2
    a.imshow(im[c:c + crop, c:c + crop]); a.set_title(p.stem, fontsize=8); a.axis('off')
plt.tight_layout(); plt.savefig(OUT / f'z{z + 2}-side-by-side.png', dpi=110); plt.show()

## 8b. Explore it in the notebook

Two viewers, at full resolution (one screen pixel per output pixel, no resampling by the browser):

- **bench**: pick a sample and two methods; the true z19 tile is always shown next to them;
- **superzoom**: pick two of the saved z20/z21 mosaics and a region of them; same region side by side.

In [ ]:
import base64, ipywidgets as W
from IPython.display import display, HTML

def html_img(a, title):
    b = io.BytesIO(); Image.fromarray((np.clip(a, 0, 1) * 255 + 0.5).astype(np.uint8)).save(b, 'PNG')
    return (f'<figure style="margin:4px;display:inline-block"><figcaption style="font:12px sans-serif">{title}</figcaption>'
            f'<img style="image-rendering:pixelated" src="data:image/png;base64,{base64.b64encode(b.getvalue()).decode()}">'
            f'</figure>')

@torch.no_grad()
def predict(m, real):
    return to_np(METHODS[m](to_t(real)).clamp(0, 1)[..., 2 * H:2 * H + TS, 2 * H:2 * H + TS])

names = list(METHODS)
w_s = W.IntSlider(0, 0, len(samples) - 1, description='sample')
w_a = W.Dropdown(options=names, value='bicubic', description='left')
w_b = W.Dropdown(options=names, value=next((n for n in names if n.endswith('+cem')), names[-1]), description='right')
w_x2 = W.Checkbox(False, description='show 2x larger')

def show_bench(s, a, b, x2):
    x, y, real, _ = samples[s]
    rows = bench[(bench.input == 'real') & (bench.x == x) & (bench.y == y)].set_index('method')
    k = 2 if x2 else 1
    big = lambda im: im.repeat(k, 0).repeat(k, 1)
    parent = to_np(up_bicubic(to_t(real[H:H + 128, H:H + 128]), 2))  # the parent as the client shows it
    cap = lambda m: f'{m} — LPIPS {rows.lpips[m]:.3f}, PSNR {rows.psnr[m]:.1f}'
    display(HTML(html_img(big(tile_rgb(z, x, y)), f'true z{z} {x},{y}') + html_img(big(parent), f'z{z - 1} parent')
                 + html_img(big(predict(a, real)), cap(a)) + html_img(big(predict(b, real)), cap(b))))

display(W.VBox([W.HBox([w_s, w_x2]), W.HBox([w_a, w_b]),
                W.interactive_output(show_bench, dict(s=w_s, a=w_a, b=w_b, x2=w_x2))]))

In [ ]:
mosaics = {p.stem: np.asarray(Image.open(p), np.float32) / 255 for p in saved}
keys = list(mosaics)
m_a = W.Dropdown(options=keys, value=keys[0], description='left')
m_b = W.Dropdown(options=keys, value=keys[min(1, len(keys) - 1)], description='right')
m_x = W.FloatSlider(0.5, min=0, max=1, step=0.01, description='x')
m_y = W.FloatSlider(0.5, min=0, max=1, step=0.01, description='y')
m_size = W.IntSlider(512, 128, 1024, 64, description='size px')

def show_mosaic(a, b, fx, fy, size):
    out = ''
    for k in (a, b):
        im = mosaics[k]
        s = min(size, im.shape[0])
        c0 = int(fx * (im.shape[1] - s)); r0 = int(fy * (im.shape[0] - s))
        out += html_img(im[r0:r0 + s, c0:c0 + s], k)
    display(HTML(out))

display(W.VBox([W.HBox([m_a, m_b]), W.HBox([m_x, m_y, m_size]),
                W.interactive_output(show_mosaic, dict(a=m_a, b=m_b, fx=m_x, fy=m_y, size=m_size))]))

## 9. Take the results home

Everything under `out/` (tables and images) zipped for download. The tile dataset stays where it is (`tiles/`, one
JPEG per tile plus `index.csv`); it is the training set of a later fine-tune on `(z, z+1)` pairs of this very source.

In [ ]:
import shutil
zipped = shutil.make_archive(str(ROOT / 'overzoom-out'), 'zip', OUT)
print(zipped)
if IN_COLAB:
    from google.colab import files
    files.download(zipped)

## 10. Check and act

To fill in after the run, from the tables **and the pictures**:

- Does a learned ×2 beat bicubic on LPIPS with the *real* parent, or only on the synthetic one?
- What does `+cem` cost in LPIPS, and what does it buy in parent PSNR?
- Which method shows the grid without a halo, and does 32 px of halo remove it?
- At z20/z21: credible ground, or painted texture? Roads, roofs, trees, fields one by one.
- Next: fine-tune the best ×2 on this dataset's `(z18, z19)` pairs; then the derived-layer skeleton
  (`gpu-tile-proxy.md`) with that model as its first client.